# 02 - Yeast Proteome Screening on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/cryptic-ip-pipeline/blob/main/notebooks/02_yeast_screening_colab.ipynb)

Runs the full `crypticip` screening workflow on the yeast AlphaFold
proteome (UP000002311). A **small `--limit` fast screen** is the safe
default; the full proteome run is **clearly marked as long-running and
high-storage** and is commented out.


## Run this first — fresh-Colab setup

Configure the variables below, then run every cell in this section in
order. Re-running is safe: the clone is updated in place and pip
installs are idempotent.

- `REPO_URL` / `BRANCH` — where to fetch the code from.
- `PROJECT_DIR` — where to clone into (`/content/...` lives on the
  Colab VM and is wiped between sessions).
- `MOUNT_DRIVE` — set `True` to mount Google Drive so outputs persist.
- `DRIVE_RESULTS` — if mounting, where on Drive to put `results/`.


In [ ]:
REPO_URL    = 'https://github.com/Tommaso-R-Marena/cryptic-ip-pipeline.git'
BRANCH      = 'main'
PROJECT_DIR = '/content/cryptic-ip-pipeline'
MOUNT_DRIVE = False                  # True to mount Google Drive
DRIVE_ROOT  = '/content/drive'
DRIVE_RESULTS = '/content/drive/MyDrive/crypticip/results'  # used only if MOUNT_DRIVE


In [ ]:
import os, subprocess, sys, pathlib

project = pathlib.Path(PROJECT_DIR)
project.parent.mkdir(parents=True, exist_ok=True)

def sh(cmd, check=True):
    print('$', ' '.join(cmd))
    return subprocess.run(cmd, check=check)

if (project / '.git').exists():
    sh(['git', '-C', str(project), 'fetch', '--quiet', 'origin', BRANCH])
    sh(['git', '-C', str(project), 'checkout', BRANCH])
    sh(['git', '-C', str(project), 'reset', '--hard', f'origin/{BRANCH}'])
else:
    sh(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, str(project)])

os.chdir(project)
print('cwd:', os.getcwd())
sh([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'nbformat'])


In [ ]:
# Optional: mount Drive and redirect results/ onto it so outputs survive
# a runtime swap. Safe to re-run; pre-existing local results/ is backed
# up to a timestamped sibling rather than deleted.
import datetime, pathlib, shutil

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount(DRIVE_ROOT)
    drive_path = pathlib.Path(DRIVE_RESULTS)
    drive_path.mkdir(parents=True, exist_ok=True)
    target = pathlib.Path(PROJECT_DIR) / 'results'
    if target.is_symlink():
        target.unlink()
    elif target.exists():
        backup = target.with_name(f'results.local_backup_{datetime.datetime.now():%Y%m%d_%H%M%S}')
        shutil.move(str(target), str(backup))
        print('existing results/ backed up to:', backup)
    target.symlink_to(drive_path, target_is_directory=True)
    print('results/ ->', drive_path)
else:
    print('MOUNT_DRIVE=False; outputs will live on the Colab VM only.')


In [ ]:
# Verify the CLI is wired up correctly.
!crypticip --version
!crypticip check-env
!crypticip list-configs


## (Optional) Install external scientific binaries

Vanilla Colab does **not** ship `fpocket`, `freesasa`, `pdb2pqr`, `apbs`,
or `pymol`. The CLI runs in clearly-labelled fallback mode without them
(see `crypticip check-env`). For a **real** scientific run, install via
condacolab + mamba. This **restarts the kernel** — after the restart,
you must re-run the bootstrap cell above before continuing.

Skip this section entirely if you only want to exercise the plumbing.


In [ ]:
INSTALL_EXTERNAL = False  # set True to install fpocket/apbs/pymol via condacolab

if INSTALL_EXTERNAL:
    !pip install -q condacolab
    import condacolab
    condacolab.install()   # >>> kernel restarts here <<<


In [ ]:
# Post-restart cell (only if you set INSTALL_EXTERNAL=True above).
# After the kernel restart, RE-RUN the bootstrap cells above (clone,
# pip install) before running this — Colab loses cwd and the editable
# install on restart.
INSTALL_EXTERNAL = False
if INSTALL_EXTERNAL:
    !mamba install -y -c conda-forge -c bioconda \
        fpocket freesasa pdb2pqr apbs pymol-open-source python-freesasa
    !crypticip check-env


## Download the yeast proteome

This is the big step. `--resume` is the default and makes the download
idempotent — if Colab disconnects, just re-run the same cell.

| Variant | Disk | Time on Colab |
|---|---|---|
| `--limit 50` (smoke) | ~30 MB | <1 min |
| full proteome | ~5–6 GB | 5–15 min depending on network |

Watch disk with `!df -h /content`.


In [ ]:
# Smoke variant — only extract 50 files (fast):
!crypticip download-proteome --organism yeast --resume --limit 50


In [ ]:
# Full download (LONG-RUNNING, HIGH-STORAGE — uncomment when ready):
# !crypticip download-proteome --organism yeast --resume


## Build the index


In [ ]:
!crypticip index-proteome --organism yeast


## Small fast screen first

**Always** start with `--limit` and `--workers 2` to time things on
your Colab runtime, then scale up. The numbers below assume the smoke
download above.


In [ ]:
!crypticip screen --organism yeast --mode fast --workers 2 --limit 25


## Full screen (LONG-RUNNING — commented out)

Uncomment only when you have:
- the full proteome downloaded
- external binaries installed (otherwise the screen records zero
  pockets in fallback mode)
- a paid Colab runtime or a local machine (free-tier Colab will
  almost certainly time out)

`--resume` (default) makes re-runs idempotent.


In [ ]:
# Fast mode on the full proteome (LONG-RUNNING — uncomment when ready):
# !crypticip screen --organism yeast --mode fast --workers 4 --resume
# Deeper electrostatics (slower, needs APBS):
# !crypticip screen --organism yeast --mode deep --workers 2 --resume


## Report


In [ ]:
!crypticip report --organism yeast


## PyMOL sessions for top hits


In [ ]:
# .pml session files for the top 20 candidates.
# Use --render only if PyMOL was installed in the external-tools section.
!crypticip pymol --organism yeast --top 20


## Experimental follow-up


In [ ]:
!crypticip experimental-plan --organism yeast --top 25


## What to download from Colab

If you mounted Drive in the bootstrap, outputs are already persisted.
Otherwise zip + download:


In [ ]:
import shutil, pathlib
out = shutil.make_archive('/content/yeast_results', 'zip', 'results')
print('zipped to:', out, 'MB:', round(pathlib.Path(out).stat().st_size / 1e6, 2))
try:
    from google.colab import files
    files.download(out)
except Exception as e:
    print('skipping files.download (not on Colab):', e)


## Tips for production runs

- A real screen on a workstation (16 cores, full toolchain) takes
  ~2–4 hours on yeast.
- For human (~80 GB extracted) **do not** use free-tier Colab — use
  the project `Dockerfile` (`docs/reproducibility.md`).
- Fallback-mode numbers (no fpocket / no APBS) are plumbing
  diagnostics, **not** scientific findings.
